# EpitopeCraft core-v2：从使用到扩展

这个 notebook 是一条可执行的 walkthrough，目标是回答四个问题：

1. 新框架中的 `Step`、`Artifact`、`DesignSet`、`Flow` 和 `PipelineRunner` 各自负责什么？
2. 参数如何由每个 Step 声明，并按 Step 实例名从 YAML 解析？
3. 常见任务——靶点分析、ProteinMPNN redesign、BoltzGen refold、Boltz-2 screening——如何运行？
4. 新增一个 scorer、design/refold backend 或可复用子流程时，代码应该长什么样？

本 notebook **默认运行真实 GPU backend**，没有隐藏的 dry-run 开关。请在 GPU 计算节点上使用 `BindCraft` kernel；BoltzGen 和 Boltz-2 会分别通过它们的 conda 环境运行。当前 smoke 参数约需数分钟，生产任务应在 YAML 中提高 recycling、sampling 和 diffusion sample 数量。


In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Mapping
import json
import os
import subprocess

import yaml

# Find the repository root even when Jupyter starts in a child directory.
_candidates = (Path.cwd(), *Path.cwd().parents)
REPO_ROOT = next(
    (path for path in _candidates if (path / "epitopecraft").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Start this notebook inside the EpitopeCraft repository")
os.chdir(REPO_ROOT)

job_id = os.environ.get("SLURM_JOB_ID")
default_root = (
    Path(f"/localhd/{job_id}/epitopecraft-core-v2-demo")
    if job_id
    else Path("/tmp/epitopecraft-core-v2-demo")
)
DEMO_ROOT = Path(os.environ.get("EPITOPECRAFT_DEMO_ROOT", default_root)).resolve()
DEMO_ROOT.mkdir(parents=True, exist_ok=True)

# Fail early with actionable errors if the promised GPU backend is not present.
gpu = subprocess.run(
    ["nvidia-smi", "-L"], capture_output=True, text=True, check=False
)
assert gpu.returncode == 0 and "GPU" in gpu.stdout, "No allocated NVIDIA GPU is visible"

import colabdesign  # ProteinMPNN runs in the active BindCraft kernel.

backend_checks = {
    "boltzgen": ["conda", "run", "-n", "boltzgen", "python", "-c", "import boltzgen"],
    "boltz2": ["conda", "run", "-n", "binding_affinity", "boltz", "--help"],
}
for name, command in backend_checks.items():
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    assert result.returncode == 0, f"{name} backend is unavailable: {result.stderr}"

print(gpu.stdout.strip())
print(f"repository = {REPO_ROOT}")
print(f"demo outputs = {DEMO_ROOT}")


## 1. 心智模型

数据流由 Python 描述，行为参数放在 YAML：

```text
Pipeline input
    │
    ▼
Step(instance id + typed ports + config dataclass)
    │
    ├── DesignSet: candidate、metrics、tracks、lineage
    └── Artifact: structure/sequence/selection/table/trajectory + provenance
             │
             └── StructureArtifact.reference / residue_map
                    把 backend-local chain/residue 映射到同一 design reference
```

几个重要边界：

- `Step.id` 是实例级命名空间。一个 Pipeline 可初始化同一种 Step 多次，例如 `initial.refold` 和 `final.refold`。
- Step 只声明它消费的参数；Pipeline 汇总实际存在的 Step，拒绝无人消费的 YAML 参数。
- `Pipeline` 只描述 DAG；`PipelineRunner` 执行、缓存、保存 manifest。
- `Flow` 是可复用的 DAG 片段，适合“refold → score → filter”这类组合。
- backend-local 链名和 residue id 不应进入跨 Step 的接口；跨模块选择使用 canonical residue id。


## 2. StructureArtifact：一次处理链重命名和残基编号

下面构造一个很小的靶点和 binding complex。靶点原始链为 `T`，从 author residue 101 开始；模拟 backend 输出则把它改成链 `A`、从 1 开始编号。binder 是链 `B`。`ReferenceModel` 通过序列自动恢复对应关系。


In [ ]:
from epitopecraft.core.artifacts import (
    CanonicalResidueId,
    ReferenceModel,
    SelectionArtifact,
    StructureArtifact,
)

AA1_TO_3 = {
    "A": "ALA", "C": "CYS", "D": "ASP", "E": "GLU", "F": "PHE",
    "G": "GLY", "H": "HIS", "I": "ILE", "K": "LYS", "L": "LEU",
    "M": "MET", "N": "ASN", "P": "PRO", "Q": "GLN", "R": "ARG",
    "S": "SER", "T": "THR", "V": "VAL", "W": "TRP", "Y": "TYR",
}


def backbone_pdb(chains: Mapping[str, tuple[int, str, float]]) -> str:
    """Build a tiny backbone PDB from {chain: (start_residue, sequence, y_offset)}."""

    lines = []
    serial = 1
    for chain, (start_residue, sequence, y_offset) in chains.items():
        for index, one_letter in enumerate(sequence):
            residue_id = start_residue + index
            residue_name = AA1_TO_3[one_letter]
            x = index * 3.8
            for atom_name, element, offset in (
                ("N", "N", 0.0),
                ("CA", "C", 1.2),
                ("C", "C", 2.4),
                ("O", "O", 3.2),
            ):
                lines.append(
                    f"ATOM  {serial:5d} {atom_name:^4s} {residue_name:>3s} "
                    f"{chain}{residue_id:4d}    "
                    f"{x + offset:8.3f}{y_offset:8.3f}{0.0:8.3f}"
                    f"  1.00 80.00          {element:>2s}  "
                )
                serial += 1
        lines.append("TER")
    return "\n".join(lines) + "\nEND\n"


target_raw = StructureArtifact.from_text(
    "target_full",
    backbone_pdb({"T": (101, "AGS", 0.0)}),
)
reference = ReferenceModel.from_target(
    full_structure=target_raw,
    target_chains=("T",),
)
reference.add_protein_binder("ACCG", preferred_chain="B")

backend_output = StructureArtifact.from_text(
    "backend_complex",
    backbone_pdb({
        "A": (1, "AGS", 0.0),   # target T was renamed to A
        "B": (1, "ACCG", 3.0),  # binder was added
    }),
)
mapped_complex = reference.map_structure_auto(backend_output)
mapped_target = reference.map_structure(target_raw, {"T": "target:T"})

print("inferred chains:", reference.infer_chain_entities(backend_output))
print("mapping:", mapped_complex.residue_map.summary())
print("artifact:", mapped_complex.summary())


In [ ]:
debug_bundle = mapped_complex.write_debug_bundle(DEMO_ROOT / "artifact-debug")
print(debug_bundle)
print("\n".join(debug_bundle.residue_map.read_text().splitlines()[:8]))


`ReferenceModel.from_target(full_structure=..., local_structure=...)` 总是优先完整靶点，因此局部 epitope、hallucination、graft 和 refold 产生的结构都能映回同一套 target author numbering。`write_debug_bundle()` 会写出原结构、canonicalized 结构、residue map、entity 表、provenance 和 validation report。


## 3. 开发一个新 Step

一个 Step 的公共边界只有四项：

1. 一个不可变的 config dataclass；
2. typed input ports；
3. typed output ports；
4. `execute(inputs, context)`。

下面实现一个候选序列生成 Step 和一个轻量 scorer。这个 scorer 会在同一 Pipeline 中初始化两次，以展示 instance-scoped config。


In [ ]:
from epitopecraft.core.config import PipelineConfig
from epitopecraft.core.design import Design, DesignSet, ProteinCandidate
from epitopecraft.core.pipeline import (
    ExecutionContext,
    Flow,
    Handle,
    Pipeline,
    PipelineRunner,
    PortNamespace,
)
from epitopecraft.core.step import PortSpec, Step
from epitopecraft.operations.filter import MetricFilter


@dataclass(frozen=True)
class CandidateSourceConfig:
    """Sequences emitted by one candidate-source instance."""

    sequences: tuple[str, ...] = ("ACDEFGHI",)


class CandidateSource(Step):
    """Create protein candidates; replace this with a real design backend."""

    config_type = CandidateSourceConfig
    input_ports = {"target": PortSpec(StructureArtifact)}
    output_ports = {"designs": PortSpec(DesignSet)}

    def execute(
        self,
        inputs: Mapping[str, Any],
        context: ExecutionContext,
    ) -> dict[str, DesignSet]:
        """Create one Design per configured sequence."""

        designs = [
            Design(f"candidate-{index}", ProteinCandidate(sequence))
            for index, sequence in enumerate(self.config.sequences)
        ]
        return {"designs": DesignSet.from_designs(designs)}


@dataclass(frozen=True)
class SequenceQualityConfig:
    """Parameters consumed by one synthetic sequence scorer."""

    metric_prefix: str = "halu"
    target_length: int = 8
    pae_base: float = 0.15


class SequenceQuality(Step):
    """Attach deterministic demonstration metrics to protein designs."""

    config_type = SequenceQualityConfig
    input_ports = {"designs": PortSpec(DesignSet)}
    output_ports = {"designs": PortSpec(DesignSet)}

    def execute(
        self,
        inputs: Mapping[str, Any],
        context: ExecutionContext,
    ) -> dict[str, DesignSet]:
        """Score length agreement and cysteine fraction."""

        designs = inputs["designs"]
        for design in designs:
            sequence = design.candidate.sequence
            quality = max(
                0.0,
                1.0 - abs(len(sequence) - self.config.target_length)
                / self.config.target_length,
            )
            cysteine_fraction = sequence.count("C") / len(sequence)
            design.metrics[f"{self.config.metric_prefix}:pLDDT"] = quality
            design.metrics[f"{self.config.metric_prefix}:i-pAE"] = min(
                1.0, self.config.pae_base + cysteine_fraction
            )
            design.metrics[self.id] = {
                "quality": quality,
                "cysteine_fraction": cysteine_fraction,
            }
        return {"designs": designs}


### YAML 负责 variation，Step id 负责隔离

解析顺序为：

```text
dataclass defaults < profile(s) < steps.<id>.params < constructor params
```

未知 Step、未知参数和错误类型会在执行前报错。下面的 `rough_score` 与 `final_score` 是同一个 Python 类，但配置和缓存不会冲突。


In [ ]:
toy_config_text = """
profiles:
  smoke_quality:
    target_length: 8
    pae_base: 0.15

steps:
  propose:
    params:
      sequences:
        - ACDEFGHI
        - CCCCCCCC
        - ACDEF
        - ACDEFGHIK

  rough_score:
    profile: smoke_quality
    params:
      metric_prefix: halu

  after_hallu:
    params:
      recipe_path: epitopecraft/pipelines/config/default_filter.yaml
      recipe: after:hallucinate

  final_score:
    params:
      metric_prefix: final
      target_length: 9
      pae_base: 0.10
"""
toy_config_path = DEMO_ROOT / "toy-pipeline.yaml"
toy_config_path.write_text(toy_config_text)
toy_config = PipelineConfig.from_file(toy_config_path)

toy_pipeline = Pipeline(
    "toy_design",
    inputs={"target": StructureArtifact},
    config=toy_config,
)
proposed = toy_pipeline.add(
    CandidateSource("propose"),
    target=toy_pipeline.inputs.target,
)
rough = toy_pipeline.add(
    SequenceQuality("rough_score"),
    designs=proposed.designs,
)
screened = toy_pipeline.add(
    MetricFilter("after_hallu"),
    designs=rough.designs,
)
final = toy_pipeline.add(
    SequenceQuality("final_score"),
    designs=screened.passed,
)
toy_pipeline.output("designs", final.designs)
toy_pipeline.output("rejected", screened.rejected)

print(toy_pipeline.describe())


In [ ]:
print(json.dumps(toy_pipeline.config_schema(), indent=2, default=str))


`MetricFilter` 返回 `passed` 和 `rejected` 两条显式分支。下游昂贵步骤只绑定 `passed`，因此被筛掉的 design 不会继续占用 GPU。


In [ ]:
toy_runner = PipelineRunner(
    toy_pipeline,
    run_dir=DEMO_ROOT / "toy-run",
)
toy_result = toy_runner.run(target=target_raw)

def summarize_designs(designs: DesignSet) -> list[dict[str, Any]]:
    return [
        {
            "id": design.id,
            "sequence": design.candidate.sequence,
            "metrics": design.metrics,
            "parent": design.parent_id,
        }
        for design in designs
    ]

print("passed")
print(json.dumps(summarize_designs(toy_result.designs), indent=2))
print("rejected")
print(json.dumps(summarize_designs(toy_result.rejected), indent=2))


`PipelineRunner` 在 run root 下保存 instance-scoped cache 和 `run_manifest.json`。用相同输入和配置再次运行会 resume；可用 `overwrite_steps=("step_id",)` 只重跑指定 Step。缓存是可信本地状态，不应加载下载得到的 pickle。


In [ ]:
resumed_runner = PipelineRunner(toy_pipeline, run_dir=DEMO_ROOT / "toy-run")
_ = resumed_runner.run(target=target_raw)
print(json.dumps(resumed_runner.executions, indent=2))
print((DEMO_ROOT / "toy-run" / "run_manifest.json").read_text()[:1200])


## 4. 把多个 Step 封装成可复用 Flow

`Flow` 不处理科学数据；它只安装一段可复用 DAG。真实项目中可把 refold、RMSD/interface scoring、filter、relax、MD 组合成一个 validation flow，并在 redesign 前后复用。


In [ ]:
class QualityGate(Flow):
    """Reusable score → filter graph fragment."""

    def build(
        self,
        pipeline: Pipeline,
        *,
        flow_id: str,
        designs: Handle,
    ) -> PortNamespace:
        """Install namespaced child Steps and expose filter branches."""

        scored = pipeline.add(
            SequenceQuality(f"{flow_id}.score"),
            designs=designs,
        )
        return pipeline.add(
            MetricFilter(f"{flow_id}.filter"),
            designs=scored.designs,
        )


flow_config = PipelineConfig.from_mapping({
    "steps": {
        "propose": {
            "params": {"sequences": ["ACDEFGHI", "ACDEFGHIK", "CCCCCCCC"]}
        },
        "initial.score": {
            "params": {"metric_prefix": "halu", "target_length": 8}
        },
        "initial.filter": {
            "params": {"recipe": "after:hallucinate"}
        },
        "final.score": {
            "params": {"metric_prefix": "halu", "target_length": 9}
        },
        "final.filter": {
            "params": {"recipe": "after:hallucinate"}
        },
    }
})

flow_pipeline = Pipeline(
    "reusable_validation",
    inputs={"target": StructureArtifact},
    config=flow_config,
)
flow_proposed = flow_pipeline.add(
    CandidateSource("propose"),
    target=flow_pipeline.inputs.target,
)
initial_validation = flow_pipeline.use(
    QualityGate(), "initial", designs=flow_proposed.designs
)
final_validation = flow_pipeline.use(
    QualityGate(), "final", designs=initial_validation.passed
)
flow_pipeline.output("designs", final_validation.passed)

print(json.dumps(PipelineRunner(flow_pipeline).plan(), indent=2))


标准设计流程使用同一种拼装方式：

```python
components = StandardDesignComponents(
    design=my_design_step,                 # ColabDesign / BoltzGen / RFDiffusion
    after_design_filter=my_early_filter,
    graft=my_graft_step,
    initial_validation=my_validation_flow, # refold → score → filter
    redesign=my_mpnn_step,
    final_validation=my_validation_flow,   # 同一个 Flow，再用一次
)
pipeline = build_standard_design_workflow(components, config=config)
```

增加新 design、refold、OpenMM MD 或 MMPBSA 时，通常只增加 compatible Step；不修改 Runner 的 backend 分支。


## 5. 常见任务一：从已有复合物发现 binding-site patches

生产流程会先运行许多个无 hotspot design，再把最终复合物交给 `BindingSiteDiscovery`。这里复用前面的 mapped complex，演示 contact frequency、空间聚类和 motif 扩展。输出 residue 全部是 canonical id。


In [ ]:
from epitopecraft.analysis.binding_sites import BindingSiteDiscovery

site_designs = DesignSet.from_designs([
    Design(
        "pose-1",
        ProteinCandidate("ACCG"),
        artifacts={"complex": mapped_complex},
    ),
    Design(
        "pose-2",
        ProteinCandidate("ACCG"),
        artifacts={"complex": mapped_complex},
    ),
])

site_config_path = DEMO_ROOT / "site-discovery.yaml"
site_config_path.write_text("""
steps:
  binding_sites:
    params:
      structure_key: complex
      contact_distance: 5.0
      frequency_threshold: 0.5
      cluster_distance: 12.0
      minimum_patch_size: 1
      motif_sizes: [2]
""")

site_pipeline = Pipeline(
    "site_discovery",
    inputs={"target": StructureArtifact, "designs": DesignSet},
    config=PipelineConfig.from_file(site_config_path),
)
discovered = site_pipeline.add(
    BindingSiteDiscovery("binding_sites"),
    target=site_pipeline.inputs.target,
    designs=site_pipeline.inputs.designs,
)
site_pipeline.output("patches", discovered.patches)
site_pipeline.output("motifs", discovered.motifs)
site_pipeline.output("table", discovered.table)

site_result = PipelineRunner(
    site_pipeline,
    run_dir=DEMO_ROOT / "site-discovery-run",
).run(target=target_raw, designs=site_designs)

print([patch.summary() for patch in site_result.patches])
print([motif.summary() for motif in site_result.motifs])
print(json.dumps(site_result.table.rows, indent=2))


## 6. 常见任务二：ProteinMPNN remove-C redesign（真实 GPU）

这个流程接收已有 binding complex，把 binder 上原本为 cysteine 的位置交给 ProteinMPNN 重设计，然后保留显式 parent lineage。recipe 仍可使用任意重叠 selector/bias 组合；这里使用仓库中的 remove-C recipe。


In [ ]:
from epitopecraft.workflows.redesign import build_remove_cysteine_workflow

mpnn_config_path = DEMO_ROOT / "remove-c.yaml"
mpnn_config_path.write_text("""
steps:
  remove_c:
    params:
      structure_key: complex
      recipe_path: epitopecraft/pipelines/config/mpnn-rmC-recipe.json
      n_samples: 4
      max_sequences: 2
      batch_size: 4
      sampling_temperature: 0.1
      include_input: false
""")

mpnn_pipeline = build_remove_cysteine_workflow(
    config=PipelineConfig.from_file(mpnn_config_path)
)
mpnn_input = DesignSet.from_designs([
    Design(
        "complex",
        ProteinCandidate("ACCG"),
        artifacts={"complex": mapped_complex},
    )
])

print(mpnn_pipeline.describe())
mpnn_result = PipelineRunner(
    mpnn_pipeline,
    run_dir=DEMO_ROOT / "remove-c-run",
).run(designs=mpnn_input)

print([
    {
        "id": design.id,
        "parent": design.parent_id,
        "sequence": design.candidate.sequence,
        "metrics": design.metrics["remove_c"],
    }
    for design in mpnn_result.designs
])
assert all("C" not in design.candidate.sequence for design in mpnn_result.designs)


## 7. 常见任务三：BoltzGen epitope-only co-fold/refold（真实 GPU）

`BoltzGenRefold` 接收 target `StructureArtifact` 和 protein `DesignSet`，调用仓库内的 BoltzGen runtime，解析 summary，并把输出 CIF 自动映射回 target/binder reference。下面使用 smoke 级参数；生产配置通常提高 sampling 和 diffusion samples。


In [ ]:
from epitopecraft.backends.boltzgen.refold import BoltzGenRefold

wdr5_path = (REPO_ROOT / "epitopecraft/test/targets/WDR5-seg_6.pdb").resolve()
wdr5_target = StructureArtifact.from_file("wdr5_segment", wdr5_path)
boltzgen_input = DesignSet.from_designs([
    Design("boltzgen-smoke", ProteinCandidate("ACDEFGHIKL"))
])

boltzgen_config_path = DEMO_ROOT / "boltzgen-refold.yaml"
boltzgen_config_path.write_text("""
steps:
  boltz_refold:
    params:
      target_chains: [B]
      binder_chain: Z
      recycling_steps: 1
      sampling_steps: 10
      diffusion_samples: 1
      num_workers: 1
      run_analysis: false
""")

boltzgen_pipeline = Pipeline(
    "boltzgen_refold",
    inputs={"target": StructureArtifact, "designs": DesignSet},
    config=PipelineConfig.from_file(boltzgen_config_path),
)
boltzgen_folded = boltzgen_pipeline.add(
    BoltzGenRefold("boltz_refold"),
    target=boltzgen_pipeline.inputs.target,
    designs=boltzgen_pipeline.inputs.designs,
)
boltzgen_pipeline.output("designs", boltzgen_folded.designs)

boltzgen_result = PipelineRunner(
    boltzgen_pipeline,
    run_dir=DEMO_ROOT / "boltzgen-refold-run",
).run(target=wdr5_target, designs=boltzgen_input)

boltzgen_design = boltzgen_result.designs["boltzgen-smoke"]
boltzgen_artifact = boltzgen_design.artifacts["boltz_refold.structure"]
print(json.dumps(boltzgen_design.metrics["boltz_refold"], indent=2, default=str))
print("mapped residues:", boltzgen_artifact.residue_map.summary())
print("inferred entities:", boltzgen_artifact.reference.infer_chain_entities(boltzgen_artifact))


## 8. 常见任务四：已知 epitope 的蛋白/SMILES screening（真实 GPU）

`SelectionArtifact` 使用 canonical positions 表示 pocket。同一批次可混合小蛋白和 SMILES；Boltz-2 会为小分子运行 affinity head。当前 `use_target_template: false` 是推荐默认值，因为现有 Boltz PDB template reader 对局部或非连续编号结构仍不稳定；sequence + canonical pocket constraint 路径已经过 GPU 测试。


In [ ]:
from epitopecraft.core.design import SmallMoleculeCandidate
from epitopecraft.workflows.screening import build_epitope_screening_workflow

wdr5_reference = ReferenceModel.from_target(
    full_structure=wdr5_target,
    target_chains=("B",),
)
wdr5_mapped = wdr5_reference.map_structure(
    wdr5_target,
    {"B": "target:B"},
)
known_site = SelectionArtifact(
    "known_wdr5_site",
    (
        CanonicalResidueId("target:B", 1),
        CanonicalResidueId("target:B", 2),
    ),
)
screen_candidates = (
    ProteinCandidate("ACDEFGHIKL"),
    SmallMoleculeCandidate("CCO"),
)

screen_config_path = DEMO_ROOT / "boltz2-screen.yaml"
screen_config_path.write_text("""
steps:
  screen:
    params:
      target_msa: empty
      use_target_template: false
      include_affinity: true
      recycling_steps: 1
      sampling_steps: 10
      diffusion_samples: 1
      sampling_steps_affinity: 10
      diffusion_samples_affinity: 1
      num_workers: 1
      preprocessing_threads: 1
      no_kernels: true
""")

screen_pipeline = build_epitope_screening_workflow(
    config=PipelineConfig.from_file(screen_config_path)
)
print(screen_pipeline.describe())

screen_result = PipelineRunner(
    screen_pipeline,
    run_dir=DEMO_ROOT / "boltz2-screen-run",
).run(
    target=wdr5_mapped,
    site=known_site,
    candidates=screen_candidates,
)

for design in screen_result.designs:
    metric_block = design.metrics["screen"]
    structure = design.artifacts["screen.structure"]
    print({
        "id": design.id,
        "candidate_type": type(design.candidate).__name__,
        "confidence_keys": sorted(metric_block["confidence"]),
        "has_affinity": "affinity" in metric_block,
        "mapping": structure.residue_map.summary(),
    })


## 9. 新 backend / 新功能开发 checklist

以 OpenMM MD、MMPBSA、RFDiffusion 或新的 refold/scorer 为例：

1. 定义不可变 `XConfig`，只列该 Step 真正消费的参数。
2. 给每个字段稳定语义；复杂参数放 recipe 或专用 YAML，不藏在 Runner 中。
3. 声明 typed ports。能复用 `DesignSet`、`StructureArtifact`、`TrajectoryArtifact`、`TableArtifact` 时不要发明新字典协议。
4. 在 `execute()` 中从 `context.step_dir(self.id)` 获取实例级输出目录。
5. backend 改链名/编号后，用 `ReferenceModel.map_structure_auto()` 映回 canonical reference。
6. metrics 使用 `design.metrics[self.id]` 或明确的科学 namespace；逐残基数据放 `design.tracks`。
7. 衍生序列使用 `design.child(...)` 保留 lineage。
8. backend 重依赖保持 lazy import，或通过 subprocess/独立环境适配。
9. 先写 port/config/mapping/cache contract tests，再写默认执行的真实 GPU smoke test。
10. 如果是一段重复的数据流，实现 `Flow`；不要给 `PipelineRunner` 增加 backend-specific 分支。

最小骨架：

```python
@dataclass(frozen=True)
class OpenMMMDConfig:
    structure_key: str = "relax.structure"
    temperature_kelvin: float = 300.0
    duration_ns: float = 10.0

class OpenMMMD(Step):
    config_type = OpenMMMDConfig
    input_ports = {"designs": PortSpec(DesignSet)}
    output_ports = {"designs": PortSpec(DesignSet)}

    def execute(
        self,
        inputs: Mapping[str, Any],
        context: ExecutionContext,
    ) -> dict[str, DesignSet]:
        step_dir = context.step_dir(self.id)
        for design in inputs["designs"]:
            structure = design.artifacts[self.config.structure_key]
            # run OpenMM; attach TrajectoryArtifact and summary metrics
            design.artifacts[f"{self.id}.trajectory"] = trajectory
            design.metrics[self.id] = summary
        return {"designs": inputs["designs"]}
```

这个骨架不会要求修改已有 design、refold、filter 或 Runner；只要端口相容，它就能插入现有 Flow。


## 10. 调试与交付

常用检查入口：

- `pipeline.describe()`：graph wiring + effective parameter values + value source。
- `pipeline.config_schema()`：当前 Pipeline 真正接受的参数清单。
- `PipelineRunner(...).plan()`：不加载 backend 的执行顺序。
- `<run>/run_manifest.json`：实际执行、cache 命中、耗时和配置。
- `artifact.summary()` / `artifact.validate()`：结构和 residue map 摘要。
- `artifact.write_debug_bundle(...)`：可共享的结构、mapping、entity、provenance 调试包。
- CLI：`epitopecraft pipeline describe MODULE:FACTORY --config run.yaml` 和 `epitopecraft pipeline plan ...`。


In [ ]:
manifests = sorted(DEMO_ROOT.glob("*-run/run_manifest.json"))
print("run manifests")
for manifest in manifests:
    payload = json.loads(manifest.read_text())
    print(
        manifest.relative_to(DEMO_ROOT),
        "steps=", [item["step"] for item in payload["executions"]],
    )
print(f"All demo outputs remain under: {DEMO_ROOT}")
